In [ ]:
import torch.nn as nn
from torchvision import transforms, datasets
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESOLUTION = 252
DATA_ROOT = "/home/alex/internship/datasets/aqua20/data/aqua20"

In [ ]:
class simpleCNN(nn.Module):

    def __init__(self, num_classes=10, depth=6, resolution=RESOLUTION, width=128):
        super().__init__()
        layers = []
        channels = 3
        for _ in range(depth):
            layers.append(nn.Conv2d(channels, width, kernel_size=3, padding=1))
            layers.append(nn.GroupNorm(width, width, affine=True))  # ≡ InstanceNorm affine
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.AvgPool2d(kernel_size=2, stride=2))
            channels = width

        final_spatial = resolution
        for _ in range(depth):
            final_spatial //= 2  # cohérent avec le floor de AvgPool sur tailles impaires
        layers.append(nn.Flatten())
        layers.append(nn.Linear(channels * final_spatial ** 2, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
from torch.optim import Optimizer
from torch.nn import Module
from torch.nn.modules.loss import _Loss
from tqdm.notebook import tqdm 

def train_epoch(model: Module, loader: DataLoader, optimizer: Optimizer, lossfn: _Loss, device):
    model.train()
    training_loss = 0.0
    training_acc = 0
    total = 0
    for x,y in tqdm(loader, desc="train", leave=False):
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(x)
        loss = lossfn(outputs, y)
        loss.backward()
        optimizer.step()
        training_loss += loss.item() * x.size(0)
        preds = outputs.argmax(dim=1)
        training_acc += (preds==y).sum().item()
        total += x.size(0)

    return training_loss/total, training_acc/total

In [ ]:
def evaluate(model: Module, loader: DataLoader, lossfn: _Loss, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            outputs = model(x)
            loss = lossfn(outputs, y)
            running_loss += loss.item() * x.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += x.size(0)

    avg_eval_loss = running_loss / total
    acc = correct/total
    return avg_eval_loss, acc

In [ ]:
transform = transforms.Compose([
    transforms.Resize(RESOLUTION),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
test_ds    = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=transform)
test_loader = DataLoader(test_ds,   batch_size=64, shuffle=False, num_workers=4)

## Full data

In [ ]:
full_train = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=transform)
full_loader = DataLoader(full_train, batch_size=64, shuffle=True, num_workers=4)

In [ ]:
cnn = simpleCNN(num_classes=20, depth=5).to(DEVICE)
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
lossfn = nn.CrossEntropyLoss()

In [ ]:
print("Training...")
history_cnn = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}

for epoch in range(10):
    train_loss, train_acc = train_epoch(cnn, full_loader, optimizer, lossfn, DEVICE)
    test_loss, test_acc = evaluate(cnn, test_loader, lossfn, DEVICE)
    history_cnn['train_loss'].append(train_loss)
    history_cnn['train_acc'].append(train_acc)
    history_cnn['test_loss'].append(test_loss)
    history_cnn['test_acc'].append(test_acc)
    print(f"Epoch {epoch+1}/10 - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

### Distilled data

In [ ]:
# DISTILLED_PTH = "../logged_files/distillation/aqua20/dinov2_vitb/dinov2_vitb_distill_252_ipc1_augs10_seathru_v100_32g/data.pth"
DISTILLED_PTH = "../logged_files/distillation/aqua20/dinov2_vitb/distill_aqua20_v100_seathru/data.pth"
distilled = torch.load(DISTILLED_PTH, map_location=DEVICE)
# distilled est un tensor (N, C, H, W) ou un dict selon ton format
# adapte selon ce que run.sh sauvegarde
print(f"Distilled data keys: {distilled.keys()}")
images_d = distilled["images"].to(DEVICE)   # shape: (20, 3, 196, 196)
labels_d = distilled["labels"].to(DEVICE)
print(labels_d)
from torch.utils.data import TensorDataset
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)
images_d_norm = (images_d - mean) / std

distill_loader = DataLoader(
    TensorDataset(images_d_norm.cpu(), labels_d.cpu()),
    batch_size=20, shuffle=True
)

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision.utils as vutils

# Load
run_name = "dinov2_vitb_distill_252_ipc1_augs10_v100_32g"
data = torch.load(DISTILLED_PTH, weights_only=False)

print(f"Keys in data: {data.keys()}")

images = data["images"]  # (N, C, H, W)
print(f"Shape: {images.shape}")

# Display grid
grid = vutils.make_grid(images, nrow=4, padding=2)
plt.figure(figsize=(12, 12))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("Distilled Images")
plt.show()

In [ ]:
print(test_ds.class_to_idx)

In [ ]:
cnn_distill = simpleCNN(num_classes=20, depth=5).to(DEVICE)
optimizer = torch.optim.Adam(cnn_distill.parameters(), lr=1e-3)
lossfn = nn.CrossEntropyLoss()

In [ ]:
print("Training...")
history_cnn_distill = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}

for epoch in range(10):
    train_loss, train_acc = train_epoch(cnn_distill, distill_loader, optimizer, lossfn, DEVICE)
    test_loss, test_acc = evaluate(cnn_distill, test_loader, lossfn, DEVICE)
    history_cnn_distill['train_loss'].append(train_loss)
    history_cnn_distill['train_acc'].append(train_acc)
    history_cnn_distill['test_loss'].append(test_loss)
    history_cnn_distill['test_acc'].append(test_acc)
    print(f"Epoch {epoch+1}/10 - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

In [ ]:
print(images_d.min(), images_d.max(), images_d.mean(), images_d.std())


In [ ]:
x, _ = next(iter(test_loader))
print(x.min(), x.max(), x.mean(), x.std())

### Vit

### Distilled

In [ ]:
from torchvision.models.vision_transformer import VisionTransformer

IMG = images_d.shape[-1]   # 252 → grille 18×18 patches en /14, comme DINOv2

def build_vit(num_classes=20, image_size=IMG, patch_size=14,
              num_layers=12, num_heads=12, hidden_dim=768, mlp_dim=3072):
    assert image_size % patch_size == 0
    return VisionTransformer(image_size=image_size, patch_size=patch_size,
                             num_layers=num_layers, num_heads=num_heads,
                             hidden_dim=hidden_dim, mlp_dim=mlp_dim,
                             num_classes=num_classes)

vit_distill = build_vit().to(DEVICE)   # ViT-B/14, ~86M params
print(f"{sum(p.numel() for p in vit_distill.parameters())/1e6:.1f}M params")

In [ ]:
from torchvision.transforms import v2


aug = v2.Compose([
    v2.RandomResizedCrop(IMG, scale=(0.5, 1.0), antialias=True),
    v2.RandomHorizontalFlip(),
])

EPOCHS, WARMUP, EVAL_EVERY = 500, 20, 25
optimizer = torch.optim.AdamW(vit_distill.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    [torch.optim.lr_scheduler.LinearLR(optimizer, 0.01, 1.0, WARMUP),
     torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP)],
    milestones=[WARMUP],
)
lossfn = nn.CrossEntropyLoss()
history_vit = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

x_all, y_all = images_d_norm.to(DEVICE), labels_d.to(DEVICE)
for epoch in tqdm(range(EPOCHS)):
    vit_distill.train()
    optimizer.zero_grad()
    x = aug(x_all)                          # vues différentes à chaque epoch
    outputs = vit_distill(x)
    loss = lossfn(outputs, y_all)
    loss.backward()
    optimizer.step()
    scheduler.step()
    if (epoch + 1) % EVAL_EVERY == 0:
        test_loss, test_acc = evaluate(vit_distill, lossfn=lossfn, loader=test_loader, device=DEVICE)
        train_acc = (outputs.argmax(1) == y_all).float().mean().item()
        history_vit['train_loss'].append(loss.item()); history_vit['train_acc'].append(train_acc)
        history_vit['test_loss'].append(test_loss);    history_vit['test_acc'].append(test_acc)
        print(f"Epoch {epoch+1}/{EPOCHS} - lr {scheduler.get_last_lr()[0]:.2e} - "
              f"train {loss.item():.3f}/{train_acc:.3f} - test {test_loss:.3f}/{test_acc:.3f}")

### Full 

In [ ]:
### Full data — ViT
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(RESOLUTION, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
full_train_aug = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=train_transform)
full_loader_aug = DataLoader(full_train_aug, batch_size=32, shuffle=True,
                             num_workers=4, pin_memory=True, drop_last=True)

vit_full = build_vit().to(DEVICE)

EPOCHS, WARMUP = 100, 5
optimizer = torch.optim.AdamW(vit_full.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    [torch.optim.lr_scheduler.LinearLR(optimizer, 0.01, 1.0, WARMUP),
     torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP)],
    milestones=[WARMUP],
)
lossfn = nn.CrossEntropyLoss()
history_vit_full = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(vit_full, full_loader_aug, optimizer, lossfn, DEVICE)
    scheduler.step()
    test_loss, test_acc = evaluate(vit_full, test_loader, lossfn, DEVICE)
    history_vit_full['train_loss'].append(train_loss)
    history_vit_full['train_acc'].append(train_acc)
    history_vit_full['test_loss'].append(test_loss)
    history_vit_full['test_acc'].append(test_acc)
    print(f"Epoch {epoch+1}/{EPOCHS} - lr {scheduler.get_last_lr()[0]:.2e} - "
          f"train {train_loss:.3f}/{train_acc:.3f} - test {test_loss:.3f}/{test_acc:.3f}")